# Getting started with MemsArrayWS objects

The `MemsArrayWS` class allows getting signals from a remote antenna running a local *Megamicros Broadcast Server (MBS)* server. 

In [1]:
import numpy as np
import time
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.ws import MemsArrayWS

log.setLevel( "INFO" )

# Set server access credentials
#HOST = 'buzenval20.fr'
#HOST = 'parisparc.biimea.tech
HOST = 'localhost'
PORT = 9002

## Connecting to the remote server

Providing a *MBS* server is running at ``HOST:PORT``, one can try to connect by creating a ``MemsArrayWS`` object.

In [2]:
# Define the antenna
try:
    antenna = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )

2023-11-12 10:39:44,773 [INFO]:  .Install MemsArrayWS settings
2023-11-12 10:39:44,774 [INFO]:  .Created a new antenna
2023-11-12 10:39:44,776 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-12 10:39:44,781 [INFO]:  .Try connecting to ws://localhost:9002...
2023-11-12 10:39:44,811 [INFO]:  .Received positive answer from server
2023-11-12 10:39:44,813 [INFO]:  .Getting settings values from remote receiver...
2023-11-12 10:39:44,816 [INFO]:  .Received settings from server [ok]
2023-11-12 10:39:44,816 [INFO]:  .Set 8 available MEMs numbered from 0 to 7
2023-11-12 10:39:44,817 [INFO]:  .No analogic channels available
2023-11-12 10:39:44,822 [INFO]:  .Starting MegamicrosWS device [ready]


## Getting remote settings

By using the `settings` command, you can get the settings of the remote antenna.
If you are running in an *asyncio* environment (jupyterlab for example), use instead `async_settings()` to get up to date settings in the same cell.

In [ ]:
print( f"Available mems: {antenna.available_mems}" )

In [ ]:
# Perform an antenna settings
antenna.settings()

# Print some results
print( f"Available mems: {antenna.available_mems}" )
print( f"Available analogs: {antenna.available_analogs}" )
print( f"Default sampling frequency: {antenna.sampling_frequency} Hz" )
print( f"frame_length: {antenna.frame_length} samples" )

## Performing a selftest

It may be a good idea to run a *selftest* of the remote antenna before getting settings.
Indeed, the antenna could have been chanded or modified since the last request. 
IOn addition, when performing a `selftest()`, new settings values are sent as response of the request so that you do not have to send a new `settings()` request.

In [ ]:
# Perform an antenna selftest
antenna.selftest()

# getting some antenna settings
print( f"Available mems: {antenna.available_mems}" )
print( f"Available analogs: {antenna.available_analogs}" )
print( f"Default sampling freequency: {antenna.sampling_frequency} Hz" )

## Halting the remote server
Notice that halting the server causes the connection to be lost. 

In [ ]:
# Stop the remote server
await antenna.shutdown()

## Running

### Getting signals from some MEMs

In [ ]:
# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [1, 2],
    duration=2,
    frame_length=512,
    signal_q_size = 0,
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
i = 0
for data in antenna:
    i += 1
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"Exit from loop. Received {i} frames. Signal shape is: {np.shape( signals )}" )

## Plotting signals

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna.sampling_frequency
fig, axs = plt.subplots( antenna.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna.channels_number ):
    axs[s].plot( time, signals[:,s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

## Saving signals as wav file
Since wavfiles are audio files, you cannot save more than 2 channels.

In [ ]:
import wave

WAV_FILENAME = 'toto.wav'

# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [17, 20],
    duration=10,
    frame_length=512,
    signal_q_size = 0,
    sampling_frequency=10000
)

print( f"Sampling frequency: {antenna.sampling_frequency}" )

with  wave.open( WAV_FILENAME, mode='wb' ) as wavfile:
    wavfile.setnchannels(2)
    wavfile.setsampwidth(2)
    wavfile.setframerate( antenna.sampling_frequency )

    # Get signals
    for data in antenna:
        signal = data >> 4
        wavfile.writeframesraw( np.int16( np.reshape( signal, np.size( signal ) ) ) )


# waiting for the end of the running thread is mandatory
antenna.wait()

## Hearing signal with *pyaudio* library

Note that `signal_q_size` is set to 0, setting the internal queue to an infinite length.
This prevents breaks in the audio stream.

In [ ]:
import pyaudio

FRAME_LENGTH = 512
SAMPLING_FREQUENCY = 50000
antenna.setSamplingFrequency( SAMPLING_FREQUENCY )

# Instantiate PyAudio and initialize PortAudio system resources (1)
p = pyaudio.PyAudio()

# Open stream
stream = p.open(
    format = pyaudio.paFloat32,
    channels = 2,
    rate = int( antenna.sampling_frequency ),
    output=True,
    frames_per_buffer=FRAME_LENGTH,
)

# Start running the remote Megamicros system
antenna.run( 
    mems=[17, 18],
    duration=10,
    frame_length=FRAME_LENGTH,
    counter_skip = True,
    signal_q_size = 0
)

# Get signals
transfers_counter = 0
for data in antenna:
    signal = data >> 4

    # convert into float and normalize with MEMs sensibility
    data = ( data.astype( np.float32 ) * antenna.sensibility )

    # write into audio stream
    stream.write( data, num_frames=FRAME_LENGTH )
    transfers_counter += 1

# Close stream and release PortAudio system resources (5)
stream.close()            
p.terminate()

antenna.wait()


## Saving signals as H5 files

You can save signal in H5 file format. In this example sigansl are saved on the MBS remote server.
The antenna receive no more signals. 

In [ ]:
antenna.run(
    mems = [3, 4],
    duration=10,
    frame_length=512,
    h5_recording=True,                          # H5 recording ON
    h5_pass_through=False,                      # perform F5 recording on server
    h5_rootdir='./',                            # directory where to save file
    h5_compressing=False,                       # Use compression or not
    background_mode=False,
    signal_q_size = 0,
    sampling_frequency=50000
)
antenna.h5_start()
antenna.wait()
antenna.h5_stop()

## Getting signals yourself

In this example, signals are received using the antenna internal queue.

In [ ]:
import queue

antenna.run(
    mems = [1, 2],
    duration=2,
    frame_length=512,
    signal_q_size = 0,
)

i = 0
while True:
    try:
        data = antenna.signal_q.get( timeout=5 )
        print( f"[{i}]" )
        i += 1
        # do what you want with data...

    except queue.Empty:
        print( f"exit from loop at i={i}" )
        break

antenna.wait()

## Listening to the Megamicro remote server
By starting a *master* run on the server, you can connect to the server from others hosts and listening to the signal stream.

### Staring the master run
This call lets the remote server starting a run in the background mode.

In [ ]:

antenna.run(
    mems = [0, 1, 2, 3, 4, 5, 6, 7],
    duration=0,
    frame_length=1024,
    sampling_frequency=10000,
    signal_q_size=0,
    job='master', 
)

In [ ]:
# Define the antenna
HOST_LISTEN = '10.3.141.1'
HOST_LISTEN = 'localhost'
PORT_LISTEN = PORT
try:
    listener = MemsArrayWS( HOST_LISTEN, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )

2023-11-12 10:39:56,343 [INFO]:  .Install MemsArrayWS settings
2023-11-12 10:39:56,346 [INFO]:  .Created a new antenna
2023-11-12 10:39:56,347 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


Available MEMs: []


2023-11-12 10:39:56,352 [INFO]:  .Try connecting to ws://localhost:9002...
2023-11-12 10:39:56,359 [INFO]:  .Received positive answer from server
2023-11-12 10:39:56,359 [INFO]:  .Getting settings values from remote receiver...
2023-11-12 10:39:56,362 [INFO]:  .Received settings from server [ok]
2023-11-12 10:39:56,363 [INFO]:  .Set 8 available MEMs numbered from 0 to 7
2023-11-12 10:39:56,364 [INFO]:  .No analogic channels available
2023-11-12 10:39:56,368 [INFO]:  .Starting MegamicrosWS device [ready]


In [4]:
print( f"Available mems: {listener.available_mems}" )
print( f"Available analogs: {listener.available_analogs}" )
print( f"Default sampling frequency: {listener.sampling_frequency} Hz" )
print( f"frame_length: {listener.frame_length} samples" )

Available mems: [0, 1, 2, 3, 4, 5, 6, 7]
Available analogs: []
Default sampling frequency: 50000 Hz
frame_length: 256 samples


Run a listener job. 
Please make attention to always specified the `job` value. Otherwise your next run will start with the old `job` value. 

Beware that if you start a listening job without a limited duration, then you have no way to stop it unless you create your own thread for that... 

In [ ]:
listener.settings()
print( f"Available mems: {listener.available_mems}" )
print( f"Available analogs: {listener.available_analogs}" )
print( f"Default sampling frequency: {listener.sampling_frequency} Hz" )
print( f"frame_length: {listener.frame_length} samples" )

Available mems: [0, 1, 2, 3, 4, 5, 6, 7]
Available analogs: []
Default sampling frequency: 10000.0 Hz
frame_length: 1024 samples


2023-11-12 10:40:58,518 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-12 10:40:58,523 [INFO]:  .Connected
2023-11-12 10:40:58,525 [INFO]:  .Send settings command to server
2023-11-12 10:40:58,526 [INFO]:  .Remote server settings command successfull
2023-11-12 10:40:58,528 [INFO]:  .Install MemsArray settings
2023-11-12 10:40:58,528 [INFO]:  .Set datatype to int32 
2023-11-12 10:40:58,529 [INFO]:  .Install MemsArrayWS settings
2023-11-12 10:40:58,529 [INFO]:  .New settings:
2023-11-12 10:40:58,530 [INFO]:   > available_mems: [0, 1, 2, 3, 4, 5, 6, 7]
2023-11-12 10:40:58,531 [INFO]:   > available_analogs: []
2023-11-12 10:40:58,532 [INFO]:   > datatype: int32
2023-11-12 10:40:58,533 [INFO]:   > frame_length: 1024
2023-11-12 10:40:58,533 [INFO]:   > mems_sensibility: 7.539965556205927e-06
2023-11-12 10:40:58,534 [INFO]:   > sampling_frequency: 10000.0 Hz
2023-11-12 10:40:58,534 [INFO]:   > system_type: Mu32


In [ ]:
listener.run(
    #mems = [8, 9, 10, 11, 12, 13, 14, 15],
    mems = [3, 4],
    frame_length=1024,
    signal_q_size=0,
    duration=4,
    job='listen'
)

# Init a np.ndarray
signals = np.ndarray( (0, listener.channels_number ) )

# Get signals
for data in listener:
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
listener.wait()
print( f"Exit from loop. Received {int( np.shape( signals )[0]/listener.frame_length )} frames. Signal shape is: {np.shape( signals )}" )

In [ ]:
print( f"Available mems: {listener.available_mems}" )
print( f"Available analogs: {listener.available_analogs}" )
print( f"Default sampling frequency: {listener.sampling_frequency} Hz" )
print( f"frame_length: {listener.frame_length} samples" )

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/listener.sampling_frequency
fig, axs = plt.subplots( listener.channels_number )
fig.suptitle('Channels activity')	
for s in range( listener.channels_number ):
    axs[s].plot( time, signals[:,s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

In [ ]:
# halt master job
antenna.halt_master()

2023-11-12 10:45:53,678 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-12 10:45:53,682 [INFO]:  .Connected
2023-11-12 10:45:53,683 [INFO]:  .Send halt master command...


2023-11-12 10:45:55,607 [INFO]:  .Halt_master command completed


In [ ]:
# halt listen or run job
listener.halt()

2023-11-12 10:41:06,724 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-12 10:41:06,727 [INFO]:  .Connected
2023-11-12 10:41:06,729 [INFO]:  .Send halt command...
2023-11-12 10:41:06,731 [ERROR]: in megamicros.log (ws.py:300): Halt failed: Halt command failed on remote server: There is no listener job or run job to halt


In [ ]:
# halt server
antenna.shutdown()

2023-11-12 10:45:55,738 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-12 10:45:55,741 [INFO]:  .Connected
2023-11-12 10:45:55,742 [INFO]:  .Send shutdown command to server
2023-11-12 10:45:55,743 [INFO]:  .Remote server shutdown success
